# Fase B2 · Segmentación clásica y recorte

**Desarrollado por:** Cinthya Ramón

**Objetivo:** obtener una región por producto y recortarla, en las 3 frutas del núcleo (mango, manzana y banano).

**Entra con:** tamaño (ADR-03), canal de color (ADR-04) y preprocesamiento (ADR-05); mientras no estén, se usa la configuración base provisional. **Sale:** la combinación elegida (ADR-06).

In [ ]:
import sys, time, cv2, numpy as np, pandas as pd
sys.path.append('..')
from src.utils import leer, mostrar, dice_iou, DATA

# Configuración base provisional: se reemplaza con los ganadores de ADR-03, 04 y 05
LADO = 512                                     # ADR-03
CANAL = 'S'                                    # ADR-04
preprocesar = lambda img: cv2.medianBlur(img, 3)  # ADR-05

FRUTAS = ('mango', 'manzana', 'banano')
MASCARAS = [m for m in sorted((DATA / 'anotaciones').glob('*.png')) if m.stem.split('_')[0] in FRUTAS]
foto_de = lambda m: next((DATA / 'raw').rglob(f'{m.stem}.jpg'))
print(len(MASCARAS), 'fotos con máscara')

## Segmentación

Cuatro pasos: **máscara inicial** → **apertura + cierre** para limpiarla → **separar regiones** (componentes conectados, o watershed para frutas que se tocan) → **contornos** para recortar cada región. Se descartan regiones menores al 1 % de la imagen.

La máscara inicial sale de **Otsu** sobre el canal elegido o de **Canny**: Canny da bordes, no regiones, así que se engrosan los bordes y se rellenan sus contornos para poder medirlo con Dice.

In [ ]:
def canal(img):
    hsv, lab = cv2.cvtColor(img, cv2.COLOR_BGR2HSV), cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    return {'gris': cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 'H': hsv[..., 0], 'S': hsv[..., 1],
            'a': lab[..., 1], 'b': lab[..., 2]}[CANAL]

def mascara_canny(img, t1, t2):
    bordes = cv2.dilate(cv2.Canny(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), t1, t2), np.ones((3, 3), np.uint8))
    contornos, _ = cv2.findContours(bordes, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cv2.drawContours(np.zeros_like(bordes), contornos, -1, 255, cv2.FILLED)

def segmentar(img, k=5, metodo='componentes', canny=None):
    if canny:
        umbral = mascara_canny(img, *canny)
    else:
        _, umbral = cv2.threshold(canal(img), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if umbral[[0, -1]].mean() > 127:        # si el fondo quedó blanco, se invierte
            umbral = 255 - umbral
    ee = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    limpia = cv2.morphologyEx(cv2.morphologyEx(umbral, cv2.MORPH_OPEN, ee), cv2.MORPH_CLOSE, ee)
    if metodo == 'watershed':                   # un marcador por cada centro de fruta
        dist = cv2.distanceTransform(limpia, cv2.DIST_L2, 5)
        centros = np.uint8(dist > 0.5 * dist.max()) * 255
        _, marcas = cv2.connectedComponents(centros)
        marcas = marcas + 1
        marcas[(limpia > 0) & (centros == 0)] = 0
        regiones = cv2.watershed(img, marcas) - 1
        regiones[regiones < 1] = 0
    else:
        _, regiones = cv2.connectedComponents(limpia)
    minimo = 0.01 * limpia.size
    cajas = [cv2.boundingRect(np.uint8(regiones == r)) for r in range(1, regiones.max() + 1)
             if (regiones == r).sum() > minimo]
    return umbral, limpia, cajas

### Paso a paso en una foto

In [ ]:
img = preprocesar(leer(foto_de(MASCARAS[0]), LADO))
umbral, limpia, cajas = segmentar(img)
bordes = cv2.Canny(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 50, 150)
contornos = img.copy()
for x, y, w, h in cajas:
    cv2.rectangle(contornos, (x, y), (x + w, y + h), (0, 255, 0), 2)
recortes = [img[y:y + h, x:x + w] for x, y, w, h in cajas]
mostrar([img, canal(img), umbral, limpia, bordes, mascara_canny(img, 50, 150), contornos],
        ['original', f'canal {CANAL}', 'Otsu', 'apertura + cierre', 'Canny 50/150', 'Canny rellenado',
         f'{len(cajas)} regiones'], 'b2/01_pasos.png', columnas=4)
mostrar(recortes[:6], [f'recorte {i + 1}' for i in range(len(recortes[:6]))], 'b2/02_recortes.png')

## Experimento: 8 configuraciones

- **6** de Otsu: kernel `3x3`, `5x5`, `7x7` × componentes conectados o watershed.
- **2** de Canny + contornos: umbrales `50/150` y `100/200` (kernel `5x5`, componentes).

Todas sobre las mismas 24 fotos con máscara. Se mide Dice, IoU, tiempo por foto y error de conteo (regiones detectadas contra regiones anotadas).

In [ ]:
CONFIGS = [(k, metodo, None) for k in (3, 5, 7) for metodo in ('componentes', 'watershed')]         + [(5, 'componentes', (50, 150)), (5, 'componentes', (100, 200))]

filas = []
for m in MASCARAS:
    img = preprocesar(leer(foto_de(m), LADO))
    real = cv2.resize(cv2.imread(str(m), 0), img.shape[1::-1], interpolation=cv2.INTER_NEAREST)
    _, _, stats, _ = cv2.connectedComponentsWithStats(real)
    n_real = (stats[1:, 4] > 0.01 * real.size).sum()   # regiones anotadas, mismo mínimo del 1 %
    for k, metodo, canny in CONFIGS:
        t = time.perf_counter()
        _, pred, cajas = segmentar(img, k, metodo, canny)
        ms = (time.perf_counter() - t) * 1000
        dice, iou = dice_iou(pred, real)
        filas.append([m.stem, k, metodo, f'{canny[0]}/{canny[1]}' if canny else 'Otsu', dice, iou, ms, abs(len(cajas) - n_real)])

res = pd.DataFrame(filas, columns=['foto', 'kernel', 'metodo', 'mascara', 'dice', 'iou', 'ms', 'error_conteo'])
tabla = res.groupby(['mascara', 'kernel', 'metodo'])[['dice', 'iou', 'ms', 'error_conteo']].mean().round(3)
tabla.to_csv('../resultados/fase6_segmentacion.csv')
tabla.sort_values('dice', ascending=False)

**Regla de decisión:** gana el mayor Dice. Si la diferencia es menor a 0,02, desempata el menor error de conteo y luego el menor tiempo.

El watershed no cambia la máscara (por eso da el mismo Dice): solo separa frutas que se tocan, así que su efecto se ve en el error de conteo.

In [ ]:
mejor = tabla.dice.max()
empatadas = tabla[tabla.dice >= mejor - 0.02]
MASC, K, METODO = empatadas.sort_values(['error_conteo', 'ms']).index[0]
CANNY = None if MASC == 'Otsu' else tuple(map(int, MASC.split('/')))
print(f'Elegida: {MASC}, kernel {K}, {METODO}  ·', tabla.loc[(MASC, K, METODO)].to_dict())

## Análisis de fallos

Las 3 fotos con menor Dice usando la combinación elegida.

In [ ]:
peores = res[(res.mascara == MASC) & (res.kernel == K) & (res.metodo == METODO)].nsmallest(3, 'dice')
imgs, titulos = [], []
for f in peores.itertuples():
    m = DATA / 'anotaciones' / f'{f.foto}.png'
    img = preprocesar(leer(foto_de(m), LADO))
    imgs += [img, segmentar(img, K, METODO, CANNY)[1]]
    titulos += [f.foto, f'Dice {f.dice:.2f}']
mostrar(imgs, titulos, 'b2/03_fallos.png', columnas=2)

| Foto | Causa | Mejora |
|---|---|---|
| | | |
| | | |
| | | |

## Conclusión

- Configuración elegida:
- Dice frente a la meta de 0,7: